# 🏦 ATM Cash Demand Predictor: ML & Time-Series Training Pipeline
### End-to-End Forecasting Engine: Feature Engineering, Baselines, XGBoost, LightGBM & Explainability

**Author**: ATM Cash Demand Predictor Team  
**Dataset**: Tinkoff ATM Enriched Daily Demand Dataset (June 2017 – September 2018)  
**Objective**: Build a high-performance machine learning model to forecast daily cash withdrawal demand per ATM, targeting **>= 85% forecast accuracy** and zero future data leakage.

---
### 📋 Pipeline Architecture
1. **Environment Setup & Data Ingestion**: Auto-detects Kaggle `/kaggle/input` or local paths.
2. **Feature Engineering**: Lags ($t-1, t-2, t-3, t-7, t-14, t-28$), Rolling Statistics ($3, 7, 14, 30$ days), Calendar & Contextual interaction features.
3. **Time-Based Validation Split**: Out-of-time Train / Validation / Test sets (Zero future leakage).
4. **Baseline Benchmarks**: Naive ($t-1$), Seasonal Naive ($t-7$), Moving Average (7-day).
5. **Machine Learning Models**:
   - Random Forest Regressor
   - LightGBM Regressor (if available)
   - XGBoost Regressor (with early stopping)
6. **Model Evaluation & Leaderboard**: MAE, RMSE, MAPE, WAPE, and Accuracy metrics.
7. **Explainability**: Top feature importances & visualization.
8. **Artifact Export**: Save best model (`best_forecasting_model.joblib`), predictions, and metrics to `/kaggle/working/` for deployment.


In [ ]:
import os
import sys
import glob
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False
    lgb = None

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM installed: {HAS_LIGHTGBM}")

# Auto-detect data path (Kaggle vs Local)
possible_paths = [
    "/kaggle/input/atm-cash-demand-enriched-dataset/atm_demand_enriched.csv",
    "/kaggle/input/*/atm_demand_enriched.csv",
    "data/processed/atm_demand_enriched.csv",
    "../data/processed/atm_demand_enriched.csv",
    "data/kaggle_bundle/atm_demand_enriched.csv",
    "atm_demand_enriched.csv",
]

data_path = None
for p in possible_paths:
    matched = glob.glob(p)
    if matched:
        data_path = matched[0]
        break

if data_path is None:
    raise FileNotFoundError("Could not locate atm_demand_enriched.csv. Please upload the dataset to Kaggle!")

print(f"\nLoaded Dataset from: {data_path}")


In [ ]:
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique ATMs: {df['atm_id'].nunique()}")
print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
print(f"Total Nulls: {df.isnull().sum().sum()}")

# Preview target variable summary
print("\nTarget Variable ('demand') Distribution:")
print(df['demand'].describe().round(2))

df.head(5)


## ⚙️ Step 2: Feature Engineering (Lags, Rolling Stats & Context)
In time-series forecasting, historical lags and moving averages carry strong predictive signals.
To strictly prevent **data leakage**, all rolling features and lags are calculated using **historical data strictly prior to time $t$** (`shift(1)`).
* **Lags**: $t-1, t-2, t-3$ (short-term momentum), $t-7$ (weekly seasonality), $t-14$ (bi-weekly), $t-28$ (monthly cycle).
* **Rolling Statistics**: 3, 7, 14, 30 days moving average and standard deviation.
* **Trend & Interaction Features**: Ratio of short-term demand to weekly average ($lag_1 / rolling\_mean_7$) and momentum ($lag_1 - lag_7$).


In [ ]:
# Ensure datetime parsing and chronological ordering per ATM
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['atm_id', 'date']).reset_index(drop=True)

# Generate Lag Features per ATM
lag_days = [1, 2, 3, 7, 14, 21, 28]
for lag in lag_days:
    df[f'lag_{lag}'] = df.groupby('atm_id')['demand'].shift(lag)

# Generate Rolling Statistics (using shifted series to avoid target leakage)
for window in [3, 7, 14, 30]:
    shifted = df.groupby('atm_id')['demand'].shift(1)
    df[f'rolling_mean_{window}'] = shifted.groupby(df['atm_id']).rolling(window=window, min_periods=window).mean().reset_index(drop=True)
    df[f'rolling_std_{window}'] = shifted.groupby(df['atm_id']).rolling(window=window, min_periods=window).std().reset_index(drop=True)

# Interaction / Momentum features
df['demand_ratio_7'] = df['lag_1'] / (df['rolling_mean_7'] + 1e-4)
df['demand_diff_1_7'] = df['lag_1'] - df['lag_7']
df['demand_diff_7_14'] = df['lag_7'] - df['lag_14']

# Cyclical encoding for Day of Week and Month
df['sin_dow'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['cos_dow'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)

# Convert categorical variables to integer codes
cat_cols = ['location_type', 'city', 'weather_condition', 'risk_level']
for col in cat_cols:
    if col in df.columns:
        df[f'{col}_encoded'] = pd.factorize(df[col])[0]

# Convert booleans to int
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

# Drop warmup rows (first 28 days where lag_28 is NaN)
df_model = df.dropna(subset=['lag_28', 'rolling_mean_30']).copy()
print(f"Shape after feature engineering and dropping warmup period: {df_model.shape}")
print(f"Effective date range for modeling: {df_model['date'].min().strftime('%Y-%m-%d')} to {df_model['date'].max().strftime('%Y-%m-%d')}")


## ✂️ Step 3: Time-Based Train / Validation / Test Split
Random train/test splits cause severe **future data leakage** in time series forecasting.
We use a strict **out-of-time chronological partition**:
* **Train Set**: Earliest data up to **2018-06-30** (~10 months)
* **Validation Set**: **2018-07-01 to 2018-08-15** (46 days, used for early stopping & hyperparameter tuning)
* **Test Set**: **2018-08-16 to 2018-09-30** (46 days, unseen final holdout evaluation)


In [ ]:
# Select purely numerical features for modeling
exclude_cols = [
    'demand', 'net_demand', 'deposit_amount', 'transaction_count',
    'withdrawal_count', 'deposit_count'
]
feature_cols = [
    c for c in df_model.select_dtypes(include=np.number).columns 
    if c not in exclude_cols
]
print(f"Total features selected: {len(feature_cols)}")
print("Feature columns sample:", feature_cols[:10])

# Split by Date
train_mask = df_model['date'] <= '2018-06-30'
val_mask = (df_model['date'] >= '2018-07-01') & (df_model['date'] <= '2018-08-15')
test_mask = df_model['date'] >= '2018-08-16'

X_train = df_model.loc[train_mask, feature_cols]
y_train = df_model.loc[train_mask, 'demand']

X_val = df_model.loc[val_mask, feature_cols]
y_val = df_model.loc[val_mask, 'demand']

X_test = df_model.loc[test_mask, feature_cols]
y_test = df_model.loc[test_mask, 'demand']

print(f"Train Set: {len(X_train):,} rows ({df_model.loc[train_mask, 'date'].min().strftime('%Y-%m-%d')} to {df_model.loc[train_mask, 'date'].max().strftime('%Y-%m-%d')})")
print(f"Val Set:   {len(X_val):,} rows ({df_model.loc[val_mask, 'date'].min().strftime('%Y-%m-%d')} to {df_model.loc[val_mask, 'date'].max().strftime('%Y-%m-%d')})")
print(f"Test Set:  {len(X_test):,} rows ({df_model.loc[test_mask, 'date'].min().strftime('%Y-%m-%d')} to {df_model.loc[test_mask, 'date'].max().strftime('%Y-%m-%d')})")


In [ ]:
def calculate_metrics(y_true, y_pred, model_name="Model"):
    """Compute standard time-series evaluation metrics."""
    y_true = np.array(y_true)
    y_pred = np.maximum(np.array(y_pred), 0.0)  # Demand cannot be negative
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Weighted Absolute Percentage Error (WAPE) - industry standard for intermittent/zero demand
    total_demand = np.sum(y_true)
    wape = (np.sum(np.abs(y_true - y_pred)) / total_demand) * 100 if total_demand > 0 else 0.0
    
    # Standard MAPE (avoiding division by zero by filtering out zero demand days)
    nonzero_mask = y_true > 1.0
    mape = np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100 if np.any(nonzero_mask) else 0.0
    
    # Operational Forecast Accuracy = max(0, 100 - WAPE)
    accuracy = max(0.0, 100.0 - wape)
    
    return {
        "Model": model_name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "WAPE (%)": round(wape, 2),
        "MAPE (%)": round(mape, 2),
        "Forecast Accuracy (%)": round(accuracy, 2)
    }

leaderboard = []
print("Metrics helper defined.")


## 📊 Step 4: Baseline Benchmark Models
Before trusting any complex ML algorithm, we evaluate three baseline models to set the benchmark:
1. **Naive Persistence ($t-1$)**: Predicts yesterday's demand for today.
2. **Seasonal Naive ($t-7$)**: Predicts the same day last week.
3. **7-Day Moving Average**: Predicts the average demand over the last 7 days.


In [ ]:
# 1. Naive Persistence (Lag 1)
y_pred_naive = X_test['lag_1']
m_naive = calculate_metrics(y_test, y_pred_naive, "Baseline: Naive (t-1)")
leaderboard.append(m_naive)

# 2. Seasonal Naive (Lag 7)
y_pred_snaive = X_test['lag_7']
m_snaive = calculate_metrics(y_test, y_pred_snaive, "Baseline: Seasonal Naive (t-7)")
leaderboard.append(m_snaive)

# 3. 7-Day Moving Average
y_pred_ma7 = X_test['rolling_mean_7']
m_ma7 = calculate_metrics(y_test, y_pred_ma7, "Baseline: 7-Day Moving Avg")
leaderboard.append(m_ma7)

pd.DataFrame(leaderboard)


## 🚀 Step 5: Machine Learning Models (Random Forest, LightGBM, XGBoost)
We now train advanced non-linear regression ensembles:
1. **Random Forest Regressor**: Ensembling bagging baseline.
2. **LightGBM Regressor**: Fast gradient boosting with leaf-wise splitting.
3. **XGBoost Regressor**: Tuned gradient boosting with early stopping on validation loss.


In [ ]:
# --- 1. Random Forest Regressor ---
print("Training Random Forest Regressor...")
t0 = time.time()
rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
m_rf = calculate_metrics(y_test, y_pred_rf, "Random Forest")
leaderboard.append(m_rf)
print(f"Random Forest trained in {time.time() - t0:.2f}s")

# --- 2. LightGBM Regressor ---
if HAS_LIGHTGBM:
    print("\nTraining LightGBM Regressor...")
    t0 = time.time()
    lgb_model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    y_pred_lgb = lgb_model.predict(X_test)
    m_lgb = calculate_metrics(y_test, y_pred_lgb, "LightGBM")
    leaderboard.append(m_lgb)
    print(f"LightGBM trained in {time.time() - t0:.2f}s")
else:
    print("\nLightGBM not installed in current environment. Proceeding to XGBoost...")

# --- 3. XGBoost Regressor ---
print("\nTraining XGBoost Regressor...")
t0 = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.025,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=35
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)
y_pred_xgb = xgb_model.predict(X_test)
m_xgb = calculate_metrics(y_test, y_pred_xgb, "XGBoost (Tuned)")
leaderboard.append(m_xgb)
print(f"XGBoost trained in {time.time() - t0:.2f}s (Best Iteration: {xgb_model.best_iteration})")


## 🏆 Step 6: Evaluation Leaderboard
Comparing all baseline and machine learning models on the out-of-time test set.


In [ ]:
leaderboard_df = pd.DataFrame(leaderboard).sort_values(by="MAE").reset_index(drop=True)
print("="*70)
print("             ATM CASH DEMAND FORECASTING LEADERBOARD")
print("="*70)
print(leaderboard_df.to_string(index=False))

# Identify best model
best_model_name = leaderboard_df.iloc[0]["Model"]
best_accuracy = leaderboard_df.iloc[0]["Forecast Accuracy (%)"]
print(f"\n🎯 WINNER MODEL: {best_model_name} with {best_accuracy}% Forecast Accuracy!")


In [ ]:
# Plot Actual vs Predicted for Sample ATMs
test_results = df_model.loc[test_mask, ['atm_id', 'date', 'demand']].copy()
test_results['Predicted_Demand'] = np.maximum(y_pred_xgb, 0.0)

sample_atms = [74, 85, 90, 103]
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
axes = axes.flatten()

for idx, atm_id in enumerate(sample_atms):
    sub = test_results[test_results['atm_id'] == atm_id].sort_values('date')
    ax = axes[idx]
    ax.plot(sub['date'], sub['demand'], label='Actual Cash Demand', color='#1e88e5', linewidth=2.2)
    ax.plot(sub['date'], sub['Predicted_Demand'], label='XGBoost Forecast', color='#e53935', linestyle='--', linewidth=2.0)
    ax.set_title(f"ATM #{atm_id} - Out-of-Time Forecast (Aug-Sep 2018)", fontsize=12, fontweight='bold')
    ax.set_ylabel("Cash Demand (Dispensed)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("forecast_comparison_plot.png", dpi=150)
plt.show()


In [ ]:
# Top 15 Feature Importances from XGBoost
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(data=importance_df.head(15), x='Importance', y='Feature', palette='Blues_r')
plt.title("Top 15 Most Influential Features (XGBoost Demand Forecaster)", fontsize=14, fontweight='bold')
plt.xlabel("Relative Feature Weight (F-Score)")
plt.tight_layout()
plt.savefig("feature_importance_plot.png", dpi=150)
plt.show()


## 💾 Step 7: Export Production Artifacts
We now serialize the trained model, feature columns, metrics, and test predictions into files.
These can be downloaded from Kaggle's `/kaggle/working/` directory and dropped directly into our project's `models/` directory for FastAPI model serving!


In [ ]:
# Output directory setup
output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("models")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Save Best Trained Model
model_save_path = output_dir / "best_forecasting_model.joblib"
joblib.dump(xgb_model, model_save_path)
print(f"✅ Saved trained model to: {model_save_path} ({model_save_path.stat().st_size / 1024:.1f} KB)")

# 2. Save Feature Columns List
features_save_path = output_dir / "feature_columns.json"
with open(features_save_path, "w") as f:
    json.dump(feature_cols, f, indent=2)
print(f"✅ Saved feature columns to: {features_save_path}")

# 3. Save Evaluation Metrics
metrics_save_path = output_dir / "model_metrics.json"
leaderboard_df.to_json(metrics_save_path, orient="records", indent=2)
print(f"✅ Saved evaluation leaderboard to: {metrics_save_path}")

# 4. Save Test Predictions
pred_save_path = output_dir / "test_predictions.csv"
test_results.to_csv(pred_save_path, index=False)
print(f"✅ Saved test predictions to: {pred_save_path}")

print("\n🎉 ALL ARTIFACTS EXPORTED SUCCESSFULLY!")
print("Download these files from Kaggle to use in your local FastAPI backend & React dashboard.")
